## Импорты

In [ ]:
!pip install ultralytics

In [ ]:
!pip install ensemble-boxes

In [ ]:
!pip install -q sahi

In [ ]:
import torch
import re
import cv2
import timm
import pandas as pd
import json
import os
import random
import ast

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import albumentations as A

from ultralytics import YOLO
from pathlib import Path
from ensemble_boxes import weighted_boxes_fusion
from PIL import Image
from tqdm import tqdm
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

## Вспомогательные функции

In [ ]:
def boxes_to_norm(boxes, w, h):
    """Преобразует [x_c, y_c, w, h] в [x_min, y_min, x_max, y_max] (0-1)"""
    if len(boxes) == 0: return np.empty((0, 4))
    x_c, y_c, bw, bh = boxes.T
    return np.stack([
        np.clip((x_c - bw/2) / w, 0, 1),
        np.clip((y_c - bh/2) / h, 0, 1),
        np.clip((x_c + bw/2) / w, 0, 1),
        np.clip((y_c + bh/2) / h, 0, 1)
    ], axis=1)

In [ ]:
def norm_to_boxes(boxes_norm, w, h):
    """Преобразует [x_min, y_min, x_max, y_max] (0-1) в [x_c, y_c, w, h]"""
    x1, y1, x2, y2 = boxes_norm.T
    return np.stack([
        (x1 + x2) / 2 * w, (y1 + y2) / 2 * h,
        (x2 - x1) * w, (y2 - y1) * h
    ], axis=1)

## Функция для предсказания детекторов \
*predict_normal - просто предсказание * \
*predict_with_sahi - предсказание с SAHI*

In [ ]:
def predict_normal(model, image, conf=0.1):
    h, w = image.shape[:2]
    # Только один прогон, без флипа!
    res = model.predict(source=image, conf=conf, verbose=False, half=True, imgsz=640)[0]

    boxes = res.boxes.xywh.cpu().numpy()
    scores = res.boxes.conf.cpu().numpy().tolist()
    labels = np.zeros_like(res.boxes.cls.cpu().numpy()).tolist()

    if len(boxes) == 0:
        return np.empty((0, 4)), [], []

    final_boxes = boxes_to_norm(boxes, w, h)
    final_boxes = norm_to_boxes(final_boxes, w, h) # Возвращаем в xywh
    return final_boxes, scores, labels

In [ ]:
def predict_with_sahi(sahi_model_object, image, slice_size=640, overlap=0.25):
    result = get_sliced_prediction(
        image,
        sahi_model_object,
        slice_height=slice_size,
        slice_width=slice_size,
        overlap_height_ratio=overlap,
        overlap_width_ratio=overlap,
        verbose=0
    )

    h_img, w_img = image.shape[:2]
    boxes, scores, labels = [], [], []

    for object_prediction in result.object_prediction_list:
        bbox = object_prediction.bbox.to_voc_bbox()
        boxes.append(bbox)
        scores.append(object_prediction.score.value)
        labels.append(0)

    if len(boxes) == 0:
        return np.empty((0, 4)), [], []

    boxes = np.array(boxes)
    x1, y1, x2, y2 = boxes.T
    xc = (x1 + x2) / 2
    yc = (y1 + y2) / 2
    w = x2 - x1
    h = y2 - y1
    final_boxes = np.stack([xc, yc, w, h], axis=1)

    return final_boxes, scores, labels

## Ансамбль всех детекция WBF

In [ ]:
def ensemble_detections(models, image, weights=None, iou_thr=0.6, skip_box_thr=0.1, conf_predict=0.1):
    h, w = image.shape[:2]
    all_boxes, all_scores, all_labels = [], [], []

    for i, model in enumerate(models):
        if i == 2:
            boxes, scores, labels = predict_with_sahi(model, image)
        else:
            boxes, scores, labels = predict_normal(model, image, conf=conf_predict)

        if len(boxes) > 0:
            all_boxes.append(boxes_to_norm(boxes, w, h))
            all_scores.append(np.array(scores))
            all_labels.append(np.array(labels))
        else:
            all_boxes.append(np.empty((0, 4)))
            all_scores.append(np.array([]))
            all_labels.append(np.array([]))

    if all(len(b) == 0 for b in all_boxes):
        return []

    # Финальный WBF для всех моделей
    fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
        all_boxes, all_scores, all_labels, weights=weights,
        iou_thr=iou_thr, skip_box_thr=skip_box_thr
    )

    # Возвращаем [x_c, y_c, w, h, score]
    final_boxes = norm_to_boxes(fused_boxes, w, h)
    return np.hstack([final_boxes, fused_scores[:, None]]).tolist()

## "Верезаем" людей для классификатора

In [ ]:
def crop_people_with_margin(image, detections, expand_factor=1.):
    h_img, w_img = image.shape[:2]
    crops = []
    for det in detections:
        x_c, y_c, w, h, score = det
        x_c_norm = x_c / w_img
        y_c_norm = y_c / h_img
        w_norm = w / w_img
        h_norm = h / h_img

        # Расширяем бокс для вырезания (в пикселях)
        new_w = w * expand_factor
        new_h = h * expand_factor
        x1 = int(max(0, x_c - new_w/2))
        y1 = int(max(0, y_c - new_h/2))
        x2 = int(min(w_img, x_c + new_w/2))
        y2 = int(min(h_img, y_c + new_h/2))
        crop = image[y1:y2, x1:x2]

        crops.append({
            'box': [x_c_norm, y_c_norm, w_norm, h_norm],
            'im_crop': crop,
            'score_detect': score
        })
    return crops

## Применяет ансамбль классификаторов


In [ ]:
def remap_score(value, a=0.2, b=1.0):
    old_min, old_max = 0.4, 1.0
    return a + (value - old_min) * (b - a) / (old_max - old_min)

In [ ]:
def classify_staff_ensemble(crops_data, models, weights, transforms, device='cuda', cls_thr=0.05, temperature=2.5):
    if not crops_data: return []

    staff_results = []

    for model, transform, weight in zip(models, transforms, weights):

        # Собираем тензоры всех кропов в один батч
        batch_tensors = []
        for item in crops_data:
            crop_rgb = cv2.cvtColor(item['im_crop'], cv2.COLOR_BGR2RGB)
            tensor = transform(image=crop_rgb)['image'].unsqueeze(0).half().to(device)
            batch_tensors.append(tensor)

        batch = torch.cat(batch_tensors)

        with torch.no_grad():
            logits = model(batch)
            scaled_logits = logits / temperature
            probs = torch.softmax(scaled_logits, dim=1)[:, 1].cpu().numpy()

        # Записываем вероятности обратно в объекты
        for i, prob in enumerate(probs):
            if 'probs' not in crops_data[i]: crops_data[i]['probs'] = []
            crops_data[i]['probs'].append(prob)

    # 2. Финальное решение (ИСПРАВЛЕНО ДЛЯ KAGGLE)
    for item in crops_data:
        # Усредняем мнения ансамбля (твоих крутых 2-классовых моделей)
        final_prob = np.average(item['probs'], weights=weights)

        # Считаем итоговый скор (баланс детектора и классификатора)
        score = np.sqrt(final_prob * item['score_detect'])
        score = (final_prob ** (2/3)) * (item['score_detect'] ** (1/3))

        # УБРАЛИ ЖЕСТКИЙ СРЕЗ if final_prob > 0.5!
        # Даем метрике mAP шанс вытянуть Recall за счет неуверенных предсказаний.
        # cls_thr теперь единственный фильтр (рекомендую передавать сюда 0.01 - 0.05)
        if score > cls_thr:
            if score > 0.85:
                score = 1.
            score = remap_score(score, a=0.1, b=1.0)
            staff_results.append(item['box'] + [score])

    return staff_results

## Загрузка моделей

In [ ]:
def load_sahi_model(model_path, conf=0.1, device='cuda'):
    print(f"Загрузка SAHI модели из {model_path}")
    sahi_model = AutoDetectionModel.from_pretrained(
        model_type='yolov8',
        model_path=model_path,
        confidence_threshold=conf,
        device=device
    )
    return sahi_model

def load_detection_models(folder, model_names):
    folder = Path(folder)
    models = []

    for model_name in model_names:
        model_path = folder / model_name
        if not model_path.exists():
            raise FileNotFoundError(f"Модель {model_name} не найдена в папке {folder}")

        model = YOLO(str(model_path))
        models.append(model)
        print(f"Загружена модель: {model_name}")

    print(f"Загружено {len(models)} моделей детекции")
    return models

def load_classification_models(folder, img_sizes, models_name, mean, std):
    folder = Path(folder)
    model_paths = sorted(folder.glob("*.pth"))
    assert len(model_paths) == len(img_sizes), "Количество моделей и размеров должно совпадать"

    models = []
    f1_scores = []
    transforms = []
    n_class = 0

    for i, (path, size, model_name) in enumerate(zip(model_paths, img_sizes, models_name)):
        # Извлекаем loss из имени файла
        stem = path.stem
        match = re.search(r'_([0-9.]+)$', stem)
        if match:
            f1 = float(match.group(1))
        else:
            raise ValueError(f"Не удалось извлечь f1 из имени файла {path.name}")
        if f1 < 0.95:
          n_class = 3
        else:
          n_class = 2

        model = timm.create_model(model_name, pretrained=False, num_classes=n_class)
        state_dict = torch.load(path, map_location='cpu')
        model.load_state_dict(state_dict)
        model.to(device).half().eval()
        models.append(model)

        f1_scores.append(f1)

        transform = A.Compose([
            A.LongestMaxSize(max_size=size),
            A.PadIfNeeded(min_height=size, min_width=size, border_mode=cv2.BORDER_CONSTANT, value=0),
            A.Normalize(mean=mean[i], std=std[i]),
            A.ToTensorV2(),
        ])
        transforms.append(transform)

    # Вычисляем веса по f1.
    f1_scores = [f**2 for f in f1_scores]
    sum_f1 = sum(f1_scores)
    weights = [f/sum_f1 for f in f1_scores]

    print(f"Загружено {len(models)} моделей классификации")
    print(f"Веса: {weights}")

    return models, weights, transforms

## Визуализация решения

In [ ]:
def visualize_staff(image, staff_bboxes_norm):
    h_img, w_img = image.shape[:2]
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(image_rgb)
    ax.axis('off')

    for bbox in staff_bboxes_norm:
        x_n, y_n, w_n, h_n, score = bbox
        x_center = x_n * w_img
        y_center = y_n * h_img
        box_w = w_n * w_img
        box_h = h_n * h_img
        x = x_center - box_w / 2
        y = y_center - box_h / 2

        # Рисуем прямоугольник
        rect = patches.Rectangle((x, y), box_w, box_h,
                                 linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
        # Подпись с уверенностью
        ax.text(x, y-5, f'{score:.3f}', color='lime', fontsize=10,
                bbox=dict(facecolor='black', alpha=0.5))

    plt.tight_layout()
    plt.show()

## Загрузка всех переменных

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
detect_folder = '/content/detect_people_weight'
classify_folder_class = '/content/classification_weight'
test_path = '/content/test_images/test_images'

# КЛАССИФИКАТОРЫ
classification_model_name = ['convnext_tiny.fb_in22k_ft_in1k', 'convnext_tiny.fb_in22k_ft_in1k',
                             'tf_efficientnetv2_s.in21k_ft_in1k', 'tf_efficientnetv2_s.in21k_ft_in1k']

# Размеры для классификаторов
classifier_img_sizes = [224, 224, 300, 300]

# Параметры классификаторов
m = [[0.485, 0.456, 0.406], [0.485, 0.456, 0.406], [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]]
s = [[0.229, 0.224, 0.225], [0.229, 0.224, 0.225], [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]]

# Загрузка классификаторов
clf_models, clf_weights, clf_transforms = load_classification_models(classify_folder_class,
                                                                     classifier_img_sizes,
                                                                     classification_model_name,
                                                                     mean=m,
                                                                     std=s)

# ДЕТЕКТОРЫ
detector_name = ['yolo26l.pt', 'rt-detr-l.pt']
# Веса детекторов (mAP50-95)
mAP_score = [0.734, 0.730, 0.71]

# Загрузка детекторов
det_models = load_detection_models(detect_folder, detector_name)
# Загрузка детектора SAHI
sahi_model_obj = load_sahi_model("/content/detect_people_weight/YOLOv8m_SAHI.pt")
det_models.append(sahi_model_obj)

## Проверка работоспособности

In [ ]:
# Загрузка изображения
files = os.listdir(test_path)
random_image_name = random.choice(files)
image_path = os.path.join(test_path, random_image_name)
print(image_path)
image = cv2.imread(image_path)

# 1) детекция людей
final_detections = ensemble_detections(det_models, image, weights=mAP_score, iou_thr=0.42, skip_box_thr= 0.3, conf_predict=0.1)

# 2) "вырезание" людей
people_crops = crop_people_with_margin(image, final_detections, expand_factor=1.05)

# 3) классификация
staff_bboxes = classify_staff_ensemble(people_crops,
                                      clf_models,
                                      clf_weights,
                                      clf_transforms,
                                      device='cuda',
                                      cls_thr=0.4,
                                      temperature=1)

# Вывод результатов
print(f"Найдено сотрудников: {len(staff_bboxes)}")
for bbox in staff_bboxes:
    print(f"bbox (norm): {bbox[:4]}, score: {bbox[4]:.4f}")

visualize_staff(image, staff_bboxes)

## Функция предсказания

In [ ]:
def process_test_images_and_save_txt(
    test_images_dir,
    det_models,
    clf_models,
    clf_weights,
    clf_transforms,
    preds_dir,
    expand_factor=1.1,
    iou_thr=0.6,
    skip_box_thr=0.001,
    conf = 0.1,
    score_limit=0.5,
    temp=2,
    device='cuda'):

    preds_dir = Path(preds_dir)
    os.makedirs(preds_dir, exist_ok=True)
    image_paths = sorted(Path(test_images_dir).glob('*.*'))
    valid_extensions = ('.jpg')
    image_paths = [p for p in image_paths if p.suffix.lower() in valid_extensions]

    print(f"Найдено {len(image_paths)} тестовых изображений для обработки")

    for img_path in tqdm(image_paths, desc="Обработка тестовых изображений"):
        image = cv2.imread(str(img_path))
        if image is None:
            print(f"Не удалось загрузить {img_path}!")
            continue

        # Детекция людей
        final_detections = ensemble_detections(det_models,
                                               image,
                                               weights=mAP_score,
                                               iou_thr=iou_thr,
                                               skip_box_thr=skip_box_thr,
                                               conf_predict=conf)
        # Вырезание
        people_crops = crop_people_with_margin(image,
                                               final_detections,
                                               expand_factor=expand_factor)
        # Классификация
        staff_bboxes = classify_staff_ensemble(people_crops,
                                      clf_models,
                                      clf_weights,
                                      clf_transforms,
                                      device='cuda',
                                      cls_thr=score_limit,
                                      temperature=temp)

        # Сохраняем в txt
        pred_file = preds_dir / (img_path.stem + '.txt')
        with open(pred_file, 'w') as f:
            for bbox in staff_bboxes:
                # bbox: [x_norm, y_norm, w_norm, h_norm, score]
                line = f"1 {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f} {bbox[4]:.6f}\n"
                f.write(line)
    print(f"Все предсказания сохранены в {preds_dir}")

## Функция формирования предсказание для Кегли

In [ ]:
def build_submission_from_solution_order(
    solution_csv: str,
    preds_dir: str,
    output_csv: str = "submission.csv",
    image_col: str = "image_name",
    boxes_col: str = "boxes",
    row_id_col: str = "id",
    require_score: bool = True,
    clamp_score: bool = True,
    keep_only_class: int | None = None,  # например 1, если нужны только строки класса 1; None = все классы
) -> None:
    """
    Builds submission.csv in EXACTLY the same image_name order as solution.csv.

    - Reads solution_csv, takes image_name column order as ground truth order.
    - For each image_name, looks for preds_dir/<stem>.txt
      where stem is Path(image_name).stem
    - If missing -> boxes=[]
    - Prediction txt lines: class xc yc w h score
    - Output boxes: JSON [[xc,yc,w,h,score], ...]
    """
    sol_path = Path(solution_csv)
    pdir = Path(preds_dir)

    if not sol_path.exists():
        raise FileNotFoundError(f"solution_csv not found: {sol_path}")
    if not pdir.exists() or not pdir.is_dir():
        raise FileNotFoundError(f"preds_dir not found or not a dir: {pdir}")

    sol = pd.read_csv(sol_path)
    if image_col not in sol.columns:
        raise ValueError(f"solution.csv must contain column '{image_col}'")

    # Keep original order from solution
    image_names = sol[image_col].astype(str).tolist()

    rows = []
    for idx, image_name in enumerate(image_names):
        stem = Path(image_name).stem
        pred_file = pdir / f"{stem}.txt"

        boxes = []
        if pred_file.exists():
            content = pred_file.read_text(encoding="utf-8", errors="replace").strip()
            if content:
                for ln in content.splitlines():
                    ln = ln.strip()
                    if not ln:
                        continue
                    parts = ln.split()
                    if require_score and len(parts) < 6:
                        continue
                    if len(parts) < 5:
                        continue

                    try:
                        cls = int(float(parts[0]))
                        if keep_only_class is not None and cls != keep_only_class:
                            continue

                        xc = float(parts[1])
                        yc = float(parts[2])
                        w = float(parts[3])
                        h = float(parts[4])
                        sc = float(parts[5]) if len(parts) >= 6 else 1.0
                    except ValueError:
                        continue

                    if clamp_score:
                        sc = 0.0 if sc < 0.0 else (1.0 if sc > 1.0 else sc)

                    boxes.append([xc, yc, w, h, sc])

        rows.append(
            {
                row_id_col: idx,
                image_col: image_name,  # EXACT match
                boxes_col: json.dumps(boxes, separators=(",", ":")),
            }
        )

    sub = pd.DataFrame(rows, columns=[row_id_col, image_col, boxes_col])
    sub.to_csv(output_csv, index=False)
    print(f"Saved: {output_csv} ({len(sub)} rows). Missing preds filled with [] in solution order.")

## Предсказываем и формируем предсказание

In [ ]:
preds_dir = '/content/predictions'
sample_sub_path = '/content/sample_sub.csv'

In [ ]:
process_test_images_and_save_txt(
    test_images_dir=test_path,
    det_models=det_models,
    clf_models=clf_models,
    clf_weights=clf_weights,
    clf_transforms=clf_transforms,
    preds_dir=preds_dir,
    expand_factor=1.05,
    iou_thr=0.42,
    skip_box_thr=0.3,
    conf=0.1,
    score_limit= 0.4,
    temp=1.,
    device=device
)

In [ ]:
build_submission_from_solution_order(
    solution_csv=sample_sub_path,
    preds_dir=preds_dir,
    output_csv='submission_final.csv',
    keep_only_class=1  
)

## Просмотр предсказание на Test

In [ ]:
def visualize_predictions_from_submission(submission_csv, test_images_dir,
                                          start_idx, end_idx,
                                          max_cols=5):
    # Читаем submission
    sub = pd.read_csv(submission_csv)

    # Определяем колонку с именами файлов
    possible_image_cols = ['image_name', 'filename', 'file', 'ImageId']
    image_col = None
    for col in possible_image_cols:
        if col in sub.columns:
            image_col = col
            break
    if image_col is None:
        raise ValueError("Не найдена колонка с именами изображений в submission.csv")

    # Определяем колонку с предсказаниями
    possible_pred_cols = ['prediction_string', 'PredictionString', 'boxes', 'predictions']
    pred_col = None
    for col in possible_pred_cols:
        if col in sub.columns:
            pred_col = col
            break
    if pred_col is None:
        raise ValueError("Не найдена колонка с предсказаниями в submission.csv")

    # Выбираем нужный диапазон
    if start_idx < 0 or end_idx > len(sub) or start_idx >= end_idx:
        raise ValueError(f"Индексы должны быть в [0, {len(sub)}) и start < end")
    subset = sub.iloc[start_idx:end_idx]
    num_images = len(subset)

    # Строим сетку
    rows = (num_images + max_cols - 1) // max_cols
    cols = min(max_cols, num_images)
    figsize = (20, 2 * (end_idx-start_idx))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if rows == 1 and cols == 1:
        axes = [axes]
    elif rows == 1 or cols == 1:
        axes = axes.flatten()
    else:
        axes = axes.flatten()

    test_images_dir = Path(test_images_dir)

    for i, (idx, row) in enumerate(subset.iterrows()):
        ax = axes[i]
        name = row[image_col]
        pred_str = row[pred_col]

        # Загружаем изображение
        img_path = test_images_dir / name
        if not img_path.exists():
            print(f"Предупреждение: файл {img_path} не найден")
            ax.imshow(np.zeros((100, 100, 3), dtype=np.uint8))
            ax.set_title(f"Missing: {name}")
            ax.axis('off')
            continue

        image = cv2.imread(str(img_path))
        if image is None:
            print(f"Предупреждение: не удалось загрузить {img_path}")
            continue
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        h_img, w_img = image.shape[:2]

        # Парсим предсказания
        boxes = []
        if pd.notna(pred_str) and pred_str.strip() and pred_str.strip() != '[]':
            try:
                # Используем ast.literal_eval для безопасного преобразования
                boxes = ast.literal_eval(pred_str)
                # Проверяем, что это список списков
                if not isinstance(boxes, list):
                    boxes = []
            except:
                print(f"Предупреждение: не удалось распарсить предсказания для {name}")
                boxes = []

        # Отрисовываем изображение и боксы
        ax.imshow(image_rgb)
        for box in boxes:
            if len(box) >= 5:
                xc, yc, w, h, score = box[:5]
                # Нормализованные координаты -> пиксели
                x = (xc - w/2) * w_img
                y = (yc - h/2) * h_img
                bw = w * w_img
                bh = h * h_img
                rect = patches.Rectangle((x, y), bw, bh,
                                         linewidth=2, edgecolor='lime', facecolor='none')
                ax.add_patch(rect)
                ax.text(x, y-5, f'{score:.2f}', color='lime', fontsize=8,
                        bbox=dict(facecolor='black', alpha=0.5))

        ax.set_title(f"{name} ({len(boxes)})")
        ax.axis('off')

    # Скрываем пустые подграфики
    for j in range(i+1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
visualize_predictions_from_submission('submission_final.csv',
                                      test_path,
                                      0, 50,
                                      max_cols=2)

## Поиск оптимальных параметров для функции предсказания

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import optuna
import shutil

In [ ]:
VAL_IMAGES_DIR = "random_dataset/images"
VAL_LABELS_DIR = "random_dataset/labels"
TEMP_PREDS_DIR = '/content/temp_val_preds/'

In [ ]:
def load_yolo_labels(txt_path, is_pred=False):
    """
    Читает txt файлы формата YOLO.
    Переводит координаты [x_c, y_c, w, h] -> [x_min, y_min, x_max, y_max].
    """
    boxes = []
    scores = []
    labels = []

    if not os.path.exists(txt_path):
        return torch.empty((0, 4)), torch.tensor([]), torch.tensor([], dtype=torch.int64)

    with open(txt_path, 'r') as f:
        for line in f.readlines():
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            cls_id = int(parts[0])
            if cls_id != 1:
                continue
            x_c, y_c, w, h = map(float, parts[1:5])

            # Перевод из центров в углы
            x_min, y_min = x_c - w / 2, y_c - h / 2
            x_max, y_max = x_c + w / 2, y_c + h / 2

            boxes.append([x_min, y_min, x_max, y_max])
            labels.append(cls_id)

            if is_pred and len(parts) == 6:
                scores.append(float(parts[5]))

    boxes_tensor = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.empty((0, 4))
    labels_tensor = torch.tensor(labels, dtype=torch.int64)
    scores_tensor = torch.tensor(scores, dtype=torch.float32) if is_pred else None

    return boxes_tensor, scores_tensor, labels_tensor

def calculate_map50_95(preds_dir, gt_dir):
    """
    Считает строгую метрику mAP@50-95.
    """
    metric = MeanAveragePrecision(iou_type="bbox", box_format="xyxy", class_metrics=False)

    gt_files = list(Path(gt_dir).glob('*.txt'))

    preds_list = []
    target_list = []

    # Идем по файлам разметки
    for gt_path in gt_files:
        stem = gt_path.stem
        pred_path = Path(preds_dir) / f"{stem}.txt"

        # Загружаем Ground Truth
        t_boxes, _, t_labels = load_yolo_labels(gt_path, is_pred=False)
        target_list.append(dict(boxes=t_boxes, labels=t_labels))

        # Загружаем предсказания (если файла нет, вернутся пустые тензоры)
        p_boxes, p_scores, p_labels = load_yolo_labels(pred_path, is_pred=True)
        preds_list.append(dict(boxes=p_boxes, scores=p_scores, labels=p_labels))

    # Вычисление метрики
    metric.update(preds_list, target_list)
    results = metric.compute()

    # mAP@50-95
    return results['map'].item()


def objective(trial):
    iou_thr = trial.suggest_float('iou_thr', 0.35, 0.65)
    skip_box_thr = trial.suggest_float('skip_box_thr', 0.1, 0.55, log=True)
    score_limit = trial.suggest_float('score_limit', 0.4, 0.6)
    temp = trial.suggest_float('temp', 1., 2.5)

    # Очищаем папку с временными предсказаниями перед новой попыткой
    if os.path.exists(TEMP_PREDS_DIR):
        shutil.rmtree(TEMP_PREDS_DIR)
    os.makedirs(TEMP_PREDS_DIR, exist_ok=True)

    print(f'\niou_thr: {iou_thr} | skip_box_thr: {skip_box_thr} | score_limit: {score_limit} | temp: {temp}')

    # 2. Вызываем твою оригинальную функцию предсказания на val-датасете
    process_test_images_and_save_txt(
        test_images_dir=VAL_IMAGES_DIR,
        det_models=det_models,
        clf_models=clf_models,
        clf_weights=clf_weights,
        clf_transforms=clf_transforms,
        preds_dir=TEMP_PREDS_DIR,
        expand_factor=1.05,
        iou_thr=iou_thr,
        skip_box_thr=skip_box_thr,
        conf=0.1,
        score_limit=score_limit,
        temp=temp,
        device=device
    )

    # 3. Считаем mAP@50-95
    map50_95 = calculate_map50_95(TEMP_PREDS_DIR, VAL_LABELS_DIR)
    print(f"--> mAP@50-95: {map50_95:.4f}")

    return map50_95

In [ ]:
study = optuna.create_study(direction='maximize', study_name="Store_Staff_mAP50_95")
study.optimize(objective, n_trials=10)

In [ ]:
print("ЛУЧШИЕ ПАРАМЕТРЫ:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")
print(f"\nЛучший mAP@50-95: {study.best_value:.4f}\n")